# Laser diode structures, modes and temperature {#sec-ld-structures}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter4/notebooks/ld-structures.ipynb){.colab-link} &nbsp;Run this chapter's code in Google Colab (no installation).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter4/notebooks/ld-structures.ipynb)

> Generated from the Quarto companion. The browser sliders of the website are not included; see `interactive_explorers.ipynb` for ipywidgets versions.

In [ ]:
# --- setup: works locally and in Google Colab ---------------------------
import sys
if 'google.colab' in sys.modules:
    import urllib.request
    urllib.request.urlretrieve('https://raw.githubusercontent.com/philippevelha/Optoelectronics-teaching/main/chapter4/laserlib.py', 'laserlib.py')
else:
    sys.path.insert(0, '..')                 # laserlib.py is in the parent folder
def ojs_define(**kw): pass                   # only used by the Quarto website
# ----------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import laserlib as lz
lz.use_style()

*Slides 92–100 (homojunction laser, output characteristic, double heterostructure, stripe geometry, buried heterostructures,
commercial devices), 101–102 (example: modes and cavity length) and 103–107 (beam characteristics, output spectra, temperature
dependence of threshold and wavelength, example on $d\lambda/dT$).*

## Key relations

The first laser diodes (Hall *et al.*, 1962, slide 93) were GaAs **homojunctions**: a degenerately doped pn junction with two
cleaved facets as mirrors (slide 92). The facets reflect $R=[(n-1)/(n+1)]^2\approx0.32$, enough because the gain is huge. But the
injected carriers spread over a diffusion length (µm) and the light is not guided, so the threshold current density was about
500 A mm⁻² at room temperature (slide 94 notes): only pulsed operation, or cryogenic cooling.

The **double heterostructure** (DH, slides 95–96) sandwiches a thin (0.1–0.2 µm) narrow-gap active layer between wide-gap
claddings. The band offsets confine the carriers, and the higher index of the active layer guides the light (the planar waveguide
of Ch. 2; carrier and photon confinement were introduced for LEDs in Ch. 3 and enter the threshold through Γ in
@sec-ld-equation). Thresholds dropped to a few A mm⁻², and in 1970
Hayashi and Panish, and Alferov's group, obtained continuous operation at room temperature. **Stripe** contacts (slide 97)
restrict the current to a few µm width; **buried heterostructures** (slides 98–99) surround the active region by low-index,
wide-gap material on all sides, which also guides the light laterally and keeps one transverse mode.

The cavity selects longitudinal modes (slides 101–102, as in @sec-gas):

$$
m\frac{\lambda_m}{2n}=L,\qquad \Delta\lambda_m\approx\frac{\lambda^2}{2n_gL},
$$

where strictly the **group index** $n_g=n-\lambda\,dn/d\lambda\approx4.3$ should replace $n$ (the slides use $n=3.6$).

The threshold rises exponentially with temperature (slide 105), and the wavelength drifts (slides 106–107):

$$
I_{th}=A\exp\!\left(\frac{T}{T_0}\right),\qquad
\frac{d\lambda_m}{dT}\approx\frac{\lambda_m}{n}\frac{dn}{dT}\ \text{(one mode)},\qquad
\frac{d\lambda_\text{peak}}{dT}\approx-\frac{\lambda^2}{hc}\frac{dE_g}{dT}\ \text{(gain peak)}.
$$

## Worked example: modes and cavity length (slides 101–102), recomputed

> AlGaAs laser, $L=200$ µm, peak at 870 nm, $n=3.6$, gain FWHM 6 nm. Find the mode number, the mode spacing and the number of
> modes. Repeat for $L=20$ µm.

In [ ]:
#| label: tbl-ld-modes
#| tbl-cap: "Slides 101–102 recomputed, with the group-index correction in the last column (n_g = 4.3 for GaAs at 870 nm)."
rows = []
for L in (200e-6, 20e-6):
    m = lz.mode_number(L, 870e-9, 3.6); dl = lz.mode_spacing(L, 870e-9, 3.6)[1]
    rows.append({"L (µm)": L*1e6, "m = 2nL/λ": f"{m:.1f}", "Δλ_m (nm)": f"{dl*1e9:.3f}", "modes in 6 nm": f"{6e-9/dl:.1f}",
                 "nearest modes (nm)": ", ".join(f"{2*3.6*L/k*1e9:.1f}" for k in (int(np.floor(m)), int(np.ceil(m)))),
                 "Δλ_m with n_g = 4.3 (nm)": f"{870e-9**2/(2*4.3*L)*1e9:.3f}"})
pd.DataFrame(rows)

The slide values are reproduced: $m=1655$, 0.526 nm and 11 modes for 200 µm; 5.26 nm and a single mode near 867.5 or 872.7 nm
for 20 µm. A 20 µm cavity is too short to give enough gain for an edge emitter, but this is exactly the idea of the VCSEL
(@sec-single-mode), whose cavity is about one wavelength long.

## Output spectrum versus current (slides 94, 104)

Below threshold a laser diode is an LED: its spectrum follows the gain curve. Above threshold the gain clamps just below the
loss, $g_{th}$. Each mode $m$ is then fed by spontaneous emission and amplified by its own net gain: its photon number is about
$S_m\propto1/(g_{th}-g_m)$. The mode at the gain peak gets closest to $g_{th}$ and takes over as the current rises.

```{ojs}
//| echo: false
viewof fs = Inputs.form({
  L: Inputs.range([100, 1000], {value: 250, step: 10, label: "cavity length L (µm)"}),
  ng: Inputs.range([3.4, 4.5], {value: 4.3, step: 0.05, label: "group index n_g"}),
  lam: Inputs.range([800, 1600], {value: 870, step: 1, label: "gain peak (nm)"}),
  fw: Inputs.range([5, 60], {value: 20, step: 1, label: "gain FWHM (nm)"}),
  I: Inputs.range([0.5, 5], {value: 1.3, step: 0.01, label: "current I/I_th"})
})
```

```{ojs}
//| echo: false
fsOut = {
  const dl = fs.lam**2/(2*fs.ng*fs.L*1e3), half = fs.fw/2, gain = l => Math.max(1 - ((l - fs.lam)/half)**2*0.5, 0);
  // gain deficit of the peak below threshold: falls as 1/(I/Ith - 1) above threshold
  const delta = fs.I < 1 ? 1 - fs.I*0.9 : 2e-3/(fs.I - 1 + 0.02);
  const modes = []; const off = 0.37;     // comb offset with respect to the gain peak
  for (let k = -80; k <= 80; k++) {
    const l = fs.lam + (k + off)*dl; if (Math.abs(l - fs.lam) > 2*fs.fw) continue;
    const gm = gain(l)*(1 - delta), S = 1/(1 - gm);
    modes.push({l, S});
  }
  const mx = d3.max(modes, d => d.S), sorted = modes.map(d => d.S).sort((a, b) => b - a);
  return {modes: modes.map(d => ({...d, P: d.S/mx})), dl, smsr: 10*Math.log10(sorted[0]/sorted[1]), nm: modes.length};
}
Plot.plot({
  width: Math.min(width, 1050), height: 280, x: {label: "wavelength (nm)"}, y: {label: "relative power (each spectrum normalised)", grid: true, domain: [0, 1.05]},
  marks: [Plot.ruleX(fsOut.modes, {x: "l", y1: 0, y2: "P", stroke: "#e34948", strokeWidth: fsOut.modes.length > 150 ? 1 : 2})]
})
```

```{ojs}
//| echo: false
html`<div class="result-box">mode spacing <b>${fsOut.dl.toFixed(3)} nm</b> · ${fs.I < 1 ? "below threshold: LED-like spectrum following the gain curve" : html`side-mode suppression ratio ≈ <b>${fsOut.smsr.toFixed(1)} dB</b>`}</div>`
```

This toy model (a fixed comb offset, parabolic gain) reproduces the narrowing of slide 104. Real Fabry–Perot lasers rarely exceed
20–30 dB side-mode suppression and hop between modes when the current or temperature changes. Single-frequency lasers need an extra
wavelength-selective element (@sec-single-mode).

## Temperature: threshold and mode hops (slides 105–107)

```{ojs}
//| echo: false
viewof tt = Inputs.form({
  T0: Inputs.range([30, 200], {value: 60, step: 5, label: "characteristic temperature T₀ (K)"}),
  T: Inputs.range([0, 80], {value: 25, step: 1, label: "case temperature (°C)"}),
  dpk: Inputs.range([0.1, 0.6], {value: 0.3, step: 0.01, label: "gain-peak shift dλ_p/dT (nm/K)"}),
  dmd: Inputs.range([0.02, 0.15], {value: 0.048, step: 0.001, label: "mode shift dλ_m/dT (nm/K)"}),
  dl: Inputs.range([0.1, 2], {value: 0.5, step: 0.05, label: "mode spacing (nm)"})
})
```

```{ojs}
//| echo: false
ttOut = {
  const Ith = T => 20*Math.exp((T - 25)/tt.T0), li = [], hop = [];
  for (const T of [0, 25, 50, tt.T]) for (let i = 0; i <= 100; i++) { const I = 150*i/100; li.push({I, P: Math.max(0, 0.5*(I - Ith(T))), T: `${T} °C`}); }
  for (let i = 0; i <= 400; i++) {
    const T = 80*i/400, pk = 870 + tt.dpk*(T - 25), base = 870 + tt.dmd*(T - 25);
    const k = Math.round((pk - base)/tt.dl); hop.push({T, lam: base + k*tt.dl, pk});
  }
  return {li, hop, Ith: Ith(tt.T)};
}
Plot.plot({
  width: Math.min(width, 500), height: 280, x: {label: "current (mA)"}, y: {label: "output (mW)", grid: true, domain: [0, 30]},
  color: {legend: true}, marks: [Plot.line(ttOut.li, {x: "I", y: "P", stroke: "T", strokeWidth: 2})]
})
```

```{ojs}
//| echo: false
Plot.plot({
  width: Math.min(width, 520), height: 280, x: {label: "temperature (°C)"}, y: {label: "wavelength (nm)", grid: true},
  marks: [Plot.line(ttOut.hop, {x: "T", y: "pk", stroke: "#c3c2b7", strokeDasharray: "4,3"}),
          Plot.line(ttOut.hop, {x: "T", y: "lam", stroke: "#e34948", strokeWidth: 2, curve: "step"}), Plot.ruleX([tt.T], {stroke: "#52514e"})]
})
```

```{ojs}
//| echo: false
html`<div class="result-box">I<sub>th</sub>(${tt.T} °C) = <b>${ttOut.Ith.toFixed(1)} mA</b> (20 mA at 25 °C; it doubles every ${(tt.T0*Math.LN2).toFixed(0)} K) ·
red: lasing mode, which follows the mode shift (slope ${tt.dmd} nm/K) and hops by one spacing whenever the gain peak (grey, ${tt.dpk} nm/K) has moved half a spacing away</div>`
```

In [ ]:
#| label: tbl-dldt
#| tbl-cap: "Slide 107 recomputed: wavelength drift of one mode of a GaAs laser at 870 nm, from the index change and (exercise of the slide) from thermal expansion."
lam, n = 870e-9, 3.6
dn_dT, expan = 2.0e-4, 6e-6
pd.DataFrame([("index change, (λ/n) dn/dT", f"{lam/n*dn_dT*1e9:.4f} nm/K", "0.048 nm/K"),
              ("thermal expansion, λ (1/L) dL/dT", f"{lam*expan*1e9:.4f} nm/K", "much smaller (exercise)"),
              ("gain peak, from dE_g/dT ≈ −0.45 meV/K", f"{lam**2/(lz.h*lz.c0)*0.45e-3*lz.qe*1e9:.3f} nm/K", "≈ 0.3 nm/K, mode hops (slide 106)")],
             columns=["mechanism", "dλ/dT", "slide"])

A single mode drifts by 0.05 nm/K, while the gain peak drifts six times faster with the bandgap. Over a 20 °C range the gain peak
moves by 6 nm, that is 10 mode spacings of a 250 µm laser: the laser must hop. For WDM channels spaced by 0.8 nm (Ch. 2),
telecom lasers sit on a thermoelectric cooler that holds the temperature to ±0.01 K.

## Beam divergence (slide 103)

The near field of an edge emitter is an ellipse about 1 µm × 3 µm. Diffraction (Ch. 1) makes the far field an ellipse rotated by
90°: the beam diverges fastest in the direction where it is narrowest. For a Gaussian beam of waist $w_0$ the full angle at half
intensity is $\theta_\text{FWHM}\approx1.18\lambda/(\pi w_0)$ (radians):

In [ ]:
#| label: tbl-divergence
#| tbl-cap: "Far-field divergence of a Gaussian near field (FWHM full angle) for typical mode sizes of an 870 nm edge emitter. Compare with θ⊥ ≈ 17–45° and θ∥ ≈ 8–15° on slide 116."
lam = 870e-9
rows = []
for name, w0 in (("perpendicular, thin DH (w₀ = 0.4 µm)", 0.4e-6), ("perpendicular, large-optical-cavity (w₀ = 0.8 µm)", 0.8e-6),
                 ("parallel, narrow ridge (w₀ = 1.5 µm)", 1.5e-6), ("parallel, broad stripe (w₀ = 3 µm)", 3e-6)):
    th = 2*np.degrees(np.arctan(np.sqrt(np.log(2)/2)*lam/(np.pi*w0)))     # full angle at half intensity
    rows.append({"direction": name, "near-field FWHM (µm)": f"{w0*np.sqrt(2*np.log(2)):.2f}", "far-field FWHM": f"{th:.0f}°"})
pd.DataFrame(rows)

A cylindrical lens or an anamorphic prism pair circularises the beam before it is coupled into a fiber (Ch. 2). VCSELs
(@sec-single-mode) emit a circular beam directly.

::: {.callout-tip collapse="true"}
## Going further
- **Measure the group index.** The FP mode spacing of a real laser gives $n_g$, not $n$. Below threshold the modes are visible in
  the spontaneous spectrum; their contrast gives the net gain (Hakki–Paoli method).
- **Why $T_0$ is low for InGaAsP.** Auger recombination and carrier leakage over small band offsets grow quickly with temperature.
  1.3 µm InGaAsP lasers have $T_0\approx50$–70 K, AlGaInAs lasers 80–100 K, quantum-dot lasers more than 200 K. Compute the threshold
  at 85 °C (the industrial specification) for each.
- **Thermal rollover.** Add self-heating $T=T_\text{case}+R_\text{th}(IV-P_o)$ to the threshold law and solve for the L–I curve.
  The output bends down at high current. Typical $R_\text{th}=50$–100 K/W.
:::

::: {.callout-note collapse="true"}
## Complements
- R. N. Hall *et al.*, "Coherent light emission from GaAs junctions", *Phys. Rev. Lett.* 9, 366 (1962).
- Zh. I. Alferov, "Nobel lecture: The double heterostructure concept and its applications", *Rev. Mod. Phys.* 73, 767 (2001).
- Kasap, sec. 4.10–4.11; Coldren *et al.*, ch. 2 and 5.
:::

## Try it yourself

1. A 1550 nm FP laser has $L=300$ µm and $n_g=3.7$. What is its mode spacing in nm and in GHz? How many modes fall in a 30 nm
   gain bandwidth?
2. With $T_0=60$ K, by what factor does the threshold rise between 25 °C and 85 °C?
3. Using the temperature panel, find the temperature range without mode hops for a 1 mm long laser.
4. Why did the first homojunction lasers work at 77 K but not at room temperature?